<a href="https://colab.research.google.com/github/VintaBytes/Ciencia-de-datos/blob/main/CienciaDeDatosConPython/CienciaDeDatos_Cap%C3%ADtulo006_Filtros_con_condiciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Capítulo 6 — Filtros con condiciones

## Breve repaso

En los capítulos anteriores aprendimos distintas formas de seleccionar partes de un `DataFrame`.

Primero trabajamos con la selección de columnas. Vimos cómo elegir una columna individual, varias columnas al mismo tiempo y cómo distinguir entre una `Series` y un `DataFrame`.

Después aprendimos a seleccionar filas usando `iloc`, una herramienta que trabaja con posiciones numéricas. Con `iloc` pudimos seleccionar una fila, varias filas consecutivas, filas no consecutivas y combinaciones de filas y columnas según su ubicación dentro del `DataFrame`.

Más adelante incorporamos `loc`, que permite seleccionar datos usando etiquetas del índice y nombres de columnas. Esa herramienta nos permitió escribir selecciones más claras cuando conocemos los nombres de las variables que queremos observar.

En este capítulo vamos a dar un paso muy importante: aprender a filtrar filas usando condiciones.

Filtrar significa quedarse solamente con los registros que cumplen cierto criterio. Por ejemplo, podríamos querer ver solo las ventas realizadas en la sucursal `"Centro"`, solo los productos de la categoría `"Libreria"`, solo las ventas con precio mayor a `1000`, o solo los registros cuya cantidad vendida supere cierto valor.

Esta forma de selección es fundamental porque permite responder preguntas más específicas sobre los datos. Ya no elegimos filas por su posición ni por su etiqueta, sino por lo que esas filas contienen.

Al finalizar este capítulo deberías poder:

- Comprender qué significa filtrar filas en un `DataFrame`.
- Crear condiciones usando columnas de Pandas.
- Interpretar una máscara booleana.
- Filtrar filas que cumplen una condición.
- Filtrar valores numéricos.
- Filtrar valores categóricos o de texto.
- Combinar filtros con selección de columnas.
- Reconocer errores frecuentes al construir condiciones.

## Dataset de trabajo

Para trabajar con filtros vamos a seguir usando el dataset de ventas de una librería. Cada fila representa una venta registrada y cada columna describe una característica de esa venta: el producto vendido, la categoría, el precio unitario, la cantidad vendida, la sucursal y la fecha.

Este dataset nos permite construir filtros simples y claros. Podemos seleccionar ventas según la sucursal, la categoría, el precio o la cantidad vendida. Así podremos concentrarnos en la lógica de los filtros sin cambiar todavía el contexto de trabajo.

In [1]:
import pandas as pd

datos = {
    "producto": [
        "Cuaderno", "Lapicera", "Mochila", "Regla", "Cartuchera",
        "Cuaderno", "Marcadores", "Carpeta", "Goma", "Lapiz"
    ],
    "categoria": [
        "Libreria", "Libreria", "Accesorios", "Libreria", "Accesorios",
        "Libreria", "Libreria", "Libreria", "Libreria", "Libreria"
    ],
    "precio": [
        1200, 350, 8500, 500, 3200,
        1250, 1800, 950, 300, 250
    ],
    "cantidad_vendida": [
        15, 40, 5, 25, 8,
        12, 10, 18, 30, 50
    ],
    "sucursal": [
        "Centro", "Centro", "Norte", "Sur", "Norte",
        "Sur", "Centro", "Norte", "Sur", "Centro"
    ],
    "fecha": [
        "2024-03-01", "2024-03-01", "2024-03-02", "2024-03-02", "2024-03-03",
        "2024-03-03", "2024-03-04", "2024-03-04", "2024-03-05", "2024-03-05"
    ]
}

df = pd.DataFrame(datos)

df

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
1,Lapicera,Libreria,350,40,Centro,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,2024-03-02
3,Regla,Libreria,500,25,Sur,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,2024-03-03
5,Cuaderno,Libreria,1250,12,Sur,2024-03-03
6,Marcadores,Libreria,1800,10,Centro,2024-03-04
7,Carpeta,Libreria,950,18,Norte,2024-03-04
8,Goma,Libreria,300,30,Sur,2024-03-05
9,Lapiz,Libreria,250,50,Centro,2024-03-05


Al mostrar el `DataFrame`, podemos recordar su estructura general. Tenemos una tabla pequeña, con 10 ventas registradas y 6 columnas.

A partir de ahora, en lugar de seleccionar filas por posición o por etiqueta, vamos a seleccionar filas según condiciones. Es decir, vamos a pedirle a Pandas que nos muestre solamente los registros que cumplen cierto criterio.

## Qué significa filtrar filas

Filtrar filas significa quedarnos solamente con los registros que cumplen una condición. En una tabla de datos, no siempre queremos trabajar con todas las filas. Muchas veces necesitamos observar un subconjunto del dataset. Por ejemplo, en nuestro caso podríamos querer ver solo las ventas de la sucursal `"Centro"`, solo los productos de la categoría `"Accesorios"`, solo las ventas con precio mayor a `1000`, o solo los registros donde la cantidad vendida sea mayor que `20`.

La idea general es sencilla: formulamos una condición y Pandas revisa fila por fila si esa condición se cumple. Las filas que cumplen la condición quedan seleccionadas. Las que no la cumplen quedan fuera del resultado.

Por ejemplo, una condición podría ser:

```text
precio mayor que 1000
```

Otra condición podría ser:

```text
sucursal igual a Centro
```

O también:

```text
cantidad vendida mayor o igual que 20
```

Este tipo de selección es muy importante porque nos permite responder preguntas más específicas. En lugar de mirar toda la tabla, podemos concentrarnos en los casos que nos interesan.

Podemos pensar el filtrado de esta manera:

```text
dataset completo → condición → filas que cumplen la condición
```

En Pandas, las condiciones se construyen a partir de columnas. Por eso, antes de filtrar necesitamos identificar qué columna contiene la información que queremos usar como criterio.

## Construir una condición

Para filtrar filas en Pandas, primero necesitamos construir una condición.

Una condición es una expresión que puede evaluarse como verdadera o falsa para cada fila del `DataFrame`.

Por ejemplo, si queremos saber qué ventas tuvieron un precio mayor que `1000`, podemos comparar la columna `precio` con el valor `1000`.

In [2]:
df["precio"] > 1000

,precio
0,True
1,False
2,True
3,False
4,True
5,True
6,True
7,False
8,False
9,False


La salida no muestra todavía las filas filtradas. Lo que vemos es una serie de valores `True` y `False`.  Cada valor indica si la condición se cumple o no para una fila del `DataFrame`.

Cuando aparece `True`, significa que esa fila cumple la condición. Cuando aparece `False`, significa que no la cumple.

En este caso, Pandas revisa cada valor de la columna `precio` y pregunta:

```text
¿este precio es mayor que 1000?
```

Si la respuesta es sí, devuelve `True`. Si la respuesta es no, devuelve `False`. Este resultado se llama **máscara booleana**.

## Qué es una máscara booleana

Una **máscara booleana** es una serie de valores `True` y `False` que tiene la misma cantidad de filas que el `DataFrame`.

En el ejemplo anterior, la condición fue:

In [3]:
df["precio"] > 1000

,precio
0,True
1,False
2,True
3,False
4,True
5,True
6,True
7,False
8,False
9,False


Pandas comparó cada valor de la columna `precio` con el número `1000` y produjo un resultado para cada fila.

Podemos guardar esa condición en una variable:

In [4]:
precios_mayores_a_1000 = df["precio"] > 1000

precios_mayores_a_1000

,precio
0,True
1,False
2,True
3,False
4,True
5,True
6,True
7,False
8,False
9,False


La variable `precios_mayores_a_1000` contiene la máscara booleana. Esta máscara no es todavía el dataset filtrado. Es solamente una serie que indica qué filas cumplen la condición.

Podemos leerla así:

```text
True  → la fila cumple la condición
False → la fila no cumple la condición
```

Esta idea es fundamental para entender los filtros en Pandas. Primero construimos una condición. Esa condición genera una máscara booleana. Luego usamos esa máscara para seleccionar las filas que queremos conservar.


```text
condición → máscara booleana → DataFrame filtrado
```

En la siguiente sección vamos a usar esta máscara para obtener solamente las ventas cuyo precio es mayor que `1000`.


## Aplicar una máscara booleana

Una vez que tenemos una máscara booleana, podemos usarla para filtrar el `DataFrame`.

La idea es pedirle a Pandas que conserve solamente las filas donde la máscara tiene el valor `True`.

En nuestro caso, ya creamos una máscara llamada `precios_mayores_a_1000`. Ahora podemos usarla dentro de los corchetes del `DataFrame`.

In [5]:
df[precios_mayores_a_1000]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,2024-03-03
5,Cuaderno,Libreria,1250,12,Sur,2024-03-03
6,Marcadores,Libreria,1800,10,Centro,2024-03-04


El resultado es un nuevo `DataFrame` que contiene solamente las ventas cuyo precio es mayor que `1000`. Las filas donde la condición era `True` quedaron incluidas. Las filas donde la condición era `False` no aparecen en el resultado.

También podemos escribir el filtro directamente, sin guardar antes la máscara en una variable:

In [6]:
df[df["precio"] > 1000]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,2024-03-03
5,Cuaderno,Libreria,1250,12,Sur,2024-03-03
6,Marcadores,Libreria,1800,10,Centro,2024-03-04


Esta forma es muy frecuente en Pandas. Podemos leerla en dos partes. Primero, la condición:

```python
df["precio"] > 1000
```

produce una máscara booleana. Luego, esa máscara se usa para filtrar el `DataFrame`:

```python
df[ ... ]
```

Entonces, la expresión completa:

```python
df[df["precio"] > 1000]
```

significa: mostrá solamente las filas de `df` donde la columna `precio` sea mayor que `1000`. Esta estructura será una de las más usadas cuando trabajemos con filtros.

## Filtros con valores numéricos

Los filtros numéricos permiten seleccionar filas según el valor de una columna numérica. En nuestro dataset tenemos dos columnas numéricas principales: `precio` y `cantidad_vendida`. Podemos construir condiciones usando operadores de comparación como mayor que, menor que, igual, distinto, mayor o igual, y menor o igual.

Por ejemplo, podemos seleccionar las ventas con precio menor que `1000`.

In [7]:
df[df["precio"] < 1000]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
1,Lapicera,Libreria,350,40,Centro,2024-03-01
3,Regla,Libreria,500,25,Sur,2024-03-02
7,Carpeta,Libreria,950,18,Norte,2024-03-04
8,Goma,Libreria,300,30,Sur,2024-03-05
9,Lapiz,Libreria,250,50,Centro,2024-03-05


En este caso, Pandas conserva solamente las filas donde el valor de la columna `precio` es menor que `1000`.

También podemos seleccionar ventas con precio mayor o igual que `1000`:

In [8]:
df[df["precio"] >= 1000]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,2024-03-03
5,Cuaderno,Libreria,1250,12,Sur,2024-03-03
6,Marcadores,Libreria,1800,10,Centro,2024-03-04


O ventas cuya cantidad vendida sea mayor o igual que `20`:

In [9]:
df[df["cantidad_vendida"] >= 20]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
1,Lapicera,Libreria,350,40,Centro,2024-03-01
3,Regla,Libreria,500,25,Sur,2024-03-02
8,Goma,Libreria,300,30,Sur,2024-03-05
9,Lapiz,Libreria,250,50,Centro,2024-03-05


Los operadores de comparación más usados son:

| Operador | Significado |
|---|---|
| `>` | Mayor que |
| `<` | Menor que |
| `>=` | Mayor o igual que |
| `<=` | Menor o igual que |
| `==` | Igual a |
| `!=` | Distinto de |

Es importante prestar atención al operador que usamos. No es lo mismo preguntar por precios mayores que `1000` que por precios mayores o iguales que `1000`. En el primer caso, el valor `1000` queda afuera. En el segundo, quedaría incluido si existiera en el dataset.

Los filtros numéricos son muy útiles para encontrar registros que superan cierto umbral, detectar valores bajos o altos, o concentrarnos en una parte específica del dataset.

## Filtros con valores categóricos

Además de filtrar columnas numéricas, también podemos filtrar columnas categóricas o de texto. En nuestro dataset, algunas columnas de este tipo son `producto`, `categoria` y `sucursal`.

Por ejemplo, si queremos ver solamente las ventas realizadas en la sucursal `"Centro"`, podemos construir una condición sobre la columna `sucursal`.

In [10]:
df[df["sucursal"] == "Centro"]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
1,Lapicera,Libreria,350,40,Centro,2024-03-01
6,Marcadores,Libreria,1800,10,Centro,2024-03-04
9,Lapiz,Libreria,250,50,Centro,2024-03-05


En este caso usamos el operador `==`, que significa “igual a”. La condición:

```python
df["sucursal"] == "Centro"
```

revisa fila por fila si el valor de la columna `sucursal` es igual a `"Centro"`. Las filas donde esa condición se cumple quedan incluidas en el resultado.

También podemos filtrar por categoría. Por ejemplo, podemos seleccionar solamente los productos de la categoría `"Accesorios"`.

In [11]:
df[df["categoria"] == "Accesorios"]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
2,Mochila,Accesorios,8500,5,Norte,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,2024-03-03


El resultado muestra únicamente las filas cuya categoría es `"Accesorios"`.

Cuando filtramos textos, debemos escribir el valor exactamente como aparece en el `DataFrame`. Pandas distingue entre mayúsculas y minúsculas, y también tiene en cuenta espacios o diferencias de escritura.

Por ejemplo, `"Centro"` no es lo mismo que `"centro"`, y `"Accesorios"` no es lo mismo que `"accesorios"`.

Si no recordamos exactamente qué valores aparecen en una columna, podemos revisarlos con `unique()`:

In [12]:
df["sucursal"].unique()

array(['Centro', 'Norte', 'Sur'], dtype=object)

In [13]:
df["categoria"].unique()

array(['Libreria', 'Accesorios'], dtype=object)

Esta revisión es útil antes de construir un filtro, especialmente cuando trabajamos con datasets reales. A veces una misma categoría puede aparecer escrita de varias formas, y eso puede afectar los resultados del análisis.

## Filtrar valores distintos

Además de seleccionar filas donde una columna sea igual a cierto valor, también podemos hacer lo contrario: seleccionar las filas donde una columna sea distinta de cierto valor. Para eso usamos el operador `!=`.

Por ejemplo, si queremos ver todas las ventas que no fueron realizadas en la sucursal `"Centro"`, podemos escribir:

In [14]:
df[df["sucursal"] != "Centro"]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
2,Mochila,Accesorios,8500,5,Norte,2024-03-02
3,Regla,Libreria,500,25,Sur,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,2024-03-03
5,Cuaderno,Libreria,1250,12,Sur,2024-03-03
7,Carpeta,Libreria,950,18,Norte,2024-03-04
8,Goma,Libreria,300,30,Sur,2024-03-05


La condición:

```python
df["sucursal"] != "Centro"
```

significa: la columna `sucursal` debe ser distinta de `"Centro"`. El resultado conserva solamente las filas donde la sucursal no es `"Centro"`.

También podríamos seleccionar los registros que no pertenecen a la categoría `"Libreria"`:


In [15]:
df[df["categoria"] != "Libreria"]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
2,Mochila,Accesorios,8500,5,Norte,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,2024-03-03


En nuestro dataset, esta selección devuelve las ventas de productos que pertenecen a otra categoría.

Este tipo de filtro es útil cuando queremos excluir un grupo del análisis. Por ejemplo, podríamos querer analizar todas las sucursales excepto una, todos los productos que no pertenecen a cierta categoría o todos los registros que no tienen determinado valor.

Como siempre, es importante revisar primero qué valores existen en la columna para evitar errores de escritura. Una diferencia de mayúsculas, minúsculas o espacios puede hacer que el filtro no devuelva el resultado esperado.

## Filtrar filas y seleccionar columnas

Cuando aplicamos un filtro, Pandas devuelve todas las columnas de las filas que cumplen la condición.

Por ejemplo:

In [16]:
df[df["sucursal"] == "Centro"]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
1,Lapicera,Libreria,350,40,Centro,2024-03-01
6,Marcadores,Libreria,1800,10,Centro,2024-03-04
9,Lapiz,Libreria,250,50,Centro,2024-03-05


Esta instrucción muestra todas las columnas de las ventas realizadas en la sucursal `"Centro"`. Pero muchas veces no necesitamos ver todas las columnas. Tal vez queremos filtrar las filas y, al mismo tiempo, mostrar solamente algunas variables.

Para eso podemos combinar un filtro con selección de columnas usando `loc`.

La estructura general es:

```python
df.loc[condición, columnas]
```

Por ejemplo, podemos seleccionar las ventas de la sucursal `"Centro"` y mostrar solo las columnas `producto`, `precio` y `cantidad_vendida`.


In [17]:
df.loc[df["sucursal"] == "Centro", ["producto", "precio", "cantidad_vendida"]]

,producto,precio,cantidad_vendida
0,Cuaderno,1200,15
1,Lapicera,350,40
6,Marcadores,1800,10
9,Lapiz,250,50


Esta instrucción puede leerse así: del `DataFrame` `df`, seleccioná las filas donde la columna `sucursal` sea igual a `"Centro"` y mostrá solamente las columnas `producto`, `precio` y `cantidad_vendida`.

También podemos filtrar por categoría y mostrar solo algunas columnas:

In [18]:
df.loc[df["categoria"] == "Accesorios", ["producto", "precio", "sucursal"]]

,producto,precio,sucursal
2,Mochila,8500,Norte
4,Cartuchera,3200,Norte


Esta forma de trabajo es muy útil porque nos permite obtener una tabla más enfocada. En lugar de ver todas las columnas del dataset, nos quedamos solamente con las que aportan información para la pregunta que estamos intentando responder.

La combinación entre filtros y selección de columnas será una herramienta central en el trabajo con Pandas.

## Filtrar valores incluidos en una lista

A veces no queremos filtrar por un único valor, sino por varios valores posibles. Por ejemplo, podríamos querer seleccionar las ventas realizadas en las sucursales `"Centro"` o `"Norte"`.

Una forma cómoda de hacerlo es usar `isin()`.

El método `isin()` permite verificar si cada valor de una columna pertenece a una lista de valores permitidos.

In [19]:
df["sucursal"].isin(["Centro", "Norte"])

,sucursal
0,True
1,True
2,True
3,False
4,True
5,False
6,True
7,True
8,False
9,True


La salida es una máscara booleana. Cada fila recibe `True` si su sucursal está dentro de la lista `["Centro", "Norte"]`, y `False` si no lo está.

Luego podemos usar esa máscara para filtrar el `DataFrame`.

In [20]:
df[df["sucursal"].isin(["Centro", "Norte"])]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
1,Lapicera,Libreria,350,40,Centro,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,2024-03-03
6,Marcadores,Libreria,1800,10,Centro,2024-03-04
7,Carpeta,Libreria,950,18,Norte,2024-03-04
9,Lapiz,Libreria,250,50,Centro,2024-03-05


El resultado conserva solamente las ventas realizadas en las sucursales `"Centro"` y `"Norte"`.

También podemos aplicar `isin()` sobre otra columna. Por ejemplo, podemos seleccionar solamente algunos productos específicos:

In [21]:
df[df["producto"].isin(["Cuaderno", "Lapicera", "Lapiz"])]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
1,Lapicera,Libreria,350,40,Centro,2024-03-01
5,Cuaderno,Libreria,1250,12,Sur,2024-03-03
9,Lapiz,Libreria,250,50,Centro,2024-03-05


Esta instrucción devuelve las filas cuyo producto es `"Cuaderno"`, `"Lapicera"` o `"Lapiz"`.

Usar `isin()` suele ser más claro que escribir muchas comparaciones separadas. En lugar de preguntar varias veces si una columna es igual a un valor, construimos una lista con los valores que nos interesan y le pedimos a Pandas que conserve las filas que pertenecen a esa lista.

Esta herramienta es especialmente útil cuando trabajamos con categorías, sucursales, provincias, productos, cursos, grupos o cualquier columna que pueda contener valores repetidos.

## Filtrar textos que contienen una palabra

En algunas situaciones queremos seleccionar filas donde una columna de texto contenga cierta palabra o fragmento.

Para eso podemos usar `.str.contains()`.

Por ejemplo, si queremos buscar productos cuyo nombre contiene la letra `"a"`, podemos escribir:

In [22]:
df[df["producto"].str.contains("a")]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
1,Lapicera,Libreria,350,40,Centro,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,2024-03-02
3,Regla,Libreria,500,25,Sur,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,2024-03-03
5,Cuaderno,Libreria,1250,12,Sur,2024-03-03
6,Marcadores,Libreria,1800,10,Centro,2024-03-04
7,Carpeta,Libreria,950,18,Norte,2024-03-04
8,Goma,Libreria,300,30,Sur,2024-03-05
9,Lapiz,Libreria,250,50,Centro,2024-03-05


La condición:

```python
df["producto"].str.contains("a")
```

revisa cada valor de la columna `producto` y devuelve `True` cuando encuentra la letra `"a"` dentro del texto.

También podemos buscar un fragmento más específico. Por ejemplo:

In [23]:
df[df["producto"].str.contains("Cuad")]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
5,Cuaderno,Libreria,1250,12,Sur,2024-03-03


En este caso, Pandas devuelve las filas donde el nombre del producto contiene el fragmento `"Cuad"`.

Debemos tener en cuenta que, por defecto, esta búsqueda distingue entre mayúsculas y minúsculas. Es decir, buscar `"a"` no es lo mismo que buscar `"A"`.

Podemos comprobarlo con este ejemplo:

In [24]:
df[df["producto"].str.contains("A")]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha


Como los nombres de productos de nuestro dataset comienzan con mayúscula, una búsqueda con `"A"` podría devolver resultados diferentes de una búsqueda con `"a"`.

Si queremos hacer una búsqueda sin distinguir entre mayúsculas y minúsculas, podemos usar el parámetro `case=False`.

In [25]:
df[df["producto"].str.contains("a", case=False)]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
1,Lapicera,Libreria,350,40,Centro,2024-03-01
2,Mochila,Accesorios,8500,5,Norte,2024-03-02
3,Regla,Libreria,500,25,Sur,2024-03-02
4,Cartuchera,Accesorios,3200,8,Norte,2024-03-03
5,Cuaderno,Libreria,1250,12,Sur,2024-03-03
6,Marcadores,Libreria,1800,10,Centro,2024-03-04
7,Carpeta,Libreria,950,18,Norte,2024-03-04
8,Goma,Libreria,300,30,Sur,2024-03-05
9,Lapiz,Libreria,250,50,Centro,2024-03-05


Este tipo de filtro es útil cuando queremos buscar palabras, fragmentos de texto, códigos o patrones simples dentro de una columna.

Más adelante vamos a trabajar con operaciones de texto con mayor detalle. Por ahora, nos alcanza con reconocer que Pandas también permite construir condiciones a partir de columnas textuales.

## Errores frecuentes al construir filtros

Al construir filtros en Pandas, es común cometer algunos errores. La mayoría no son graves, pero pueden generar mensajes difíciles de interpretar al principio.

Un error frecuente es usar un solo signo igual `=` en lugar de doble signo igual `==`.

En Python, el signo `=` se usa para asignar un valor a una variable. En cambio, `==` se usa para comparar si dos valores son iguales.

Por eso, para filtrar las ventas de la sucursal `"Centro"`, escribimos:

In [26]:
df[df["sucursal"] == "Centro"]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha
0,Cuaderno,Libreria,1200,15,Centro,2024-03-01
1,Lapicera,Libreria,350,40,Centro,2024-03-01
6,Marcadores,Libreria,1800,10,Centro,2024-03-04
9,Lapiz,Libreria,250,50,Centro,2024-03-05


y no:

In [27]:
# Esta instrucción produciría un error porque = no se usa para comparar.
# df[df["sucursal"] = "Centro"]

Otro error frecuente es olvidar las comillas cuando filtramos valores de texto. Si queremos comparar con `"Centro"`, ese valor debe escribirse como texto, entre comillas.
```

In [28]:
# Esta instrucción produciría un error porque Centro no está escrito como texto.
# df[df["sucursal"] == Centro]

También puede ocurrir que escribamos un valor de texto de una forma distinta a como aparece en el dataset. Por ejemplo:

In [29]:
df[df["sucursal"] == "centro"]

,producto,categoria,precio,cantidad_vendida,sucursal,fecha


Esta instrucción no produce un error, pero devuelve un resultado vacío, porque en nuestro dataset la sucursal está escrita como `"Centro"` con mayúscula inicial.

En estos casos, conviene revisar los valores disponibles antes de filtrar:

In [30]:
df["sucursal"].unique()

array(['Centro', 'Norte', 'Sur'], dtype=object)

También debemos prestar atención a los nombres de las columnas. Si escribimos mal el nombre de una columna, Pandas no podrá encontrarla.

In [31]:
# Esta instrucción produciría un error porque la columna se llama "sucursal", no "Sucursal".
# df[df["Sucursal"] == "Centro"]

Una buena estrategia para evitar estos errores es revisar primero los nombres de las columnas y los valores disponibles en la columna que queremos usar para filtrar.

In [32]:
df.columns

Index(['producto', 'categoria', 'precio', 'cantidad_vendida', 'sucursal',
       'fecha'],
      dtype='object')

In [33]:
df["sucursal"].unique()

array(['Centro', 'Norte', 'Sur'], dtype=object)

Los filtros son una herramienta muy poderosa, pero requieren precisión. El nombre de la columna, el operador de comparación y el valor buscado deben estar escritos correctamente para que el resultado sea el esperado.

## Resumen del capítulo

En este capítulo aprendimos a filtrar filas de un `DataFrame` usando condiciones.

Hasta ahora habíamos seleccionado columnas, filas por posición con `iloc` y datos por etiquetas con `loc`. Con los filtros dimos un paso más: empezamos a seleccionar filas según el contenido real de los datos.

Vimos que una condición en Pandas produce una máscara booleana, es decir, una serie de valores `True` y `False`. Cada valor indica si una fila cumple o no cumple la condición planteada.

Por ejemplo:

```python
df["precio"] > 1000
```

genera una máscara booleana que indica qué ventas tienen un precio mayor que `1000`.

Luego usamos esa máscara para filtrar el `DataFrame`:

```python
df[df["precio"] > 1000]
```

Esta instrucción conserva solamente las filas donde la condición es verdadera.

También trabajamos con filtros numéricos usando operadores como:

```text
>
<
>=
<=
==
!=
```

Estos operadores nos permitieron seleccionar ventas según precio o cantidad vendida.

Después aplicamos filtros sobre columnas categóricas o de texto, como `sucursal`, `categoria` y `producto`. Vimos que, al filtrar textos, debemos escribir los valores exactamente como aparecen en el dataset, respetando mayúsculas, minúsculas y espacios.

También usamos `isin()` para seleccionar filas cuyos valores pertenecen a una lista de opciones posibles:

```python
df[df["sucursal"].isin(["Centro", "Norte"])]
```

Más adelante vimos que `.str.contains()` permite construir filtros a partir de fragmentos de texto:

```python
df[df["producto"].str.contains("a", case=False)]
```

Finalmente, combinamos filtros con selección de columnas usando `loc`:

```python
df.loc[df["sucursal"] == "Centro", ["producto", "precio", "cantidad_vendida"]]
```

Esta forma de trabajo nos permite obtener resultados más enfocados: no solo filtramos las filas que cumplen una condición, sino que también elegimos qué columnas queremos ver.

La idea principal de este capítulo fue:

```text
Filtrar permite seleccionar filas según condiciones construidas a partir de los datos.
```

Los filtros son una de las herramientas más importantes de Pandas, porque permiten pasar de una tabla completa a un subconjunto específico de registros que cumplen ciertos criterios.

## Próximo paso

Ya sabemos seleccionar columnas, seleccionar filas por posición, seleccionar datos por etiquetas y filtrar filas usando condiciones simples.

El siguiente paso será aprender a combinar varias condiciones en un mismo filtro. Eso nos permitirá responder preguntas más precisas, como seleccionar ventas de una sucursal determinada cuyo precio supere cierto valor, o productos de una categoría específica con una cantidad vendida mayor que cierto límite.

Para eso vamos a trabajar con operadores lógicos, como `&`, `|` y `~`, y veremos por qué en Pandas es tan importante usar paréntesis al combinar condiciones.